In [0]:
# Ler a tabela bronze
df_bronze = spark.table("bronze_hr_attrition")


In [0]:
# Criar o DataFrame base da silver
df_silver = df_bronze



In [0]:
# converter tipos
from pyspark.sql.functions import when, col

df_silver = (
    df_silver
        # Age como inteiro
        .withColumn("Age", col("Age").cast("int"))

        # Attrition como boolean
        .withColumn("Attrition", when(col("Attrition") == "Yes", True).otherwise(False))

        # OverTime como boolean
        .withColumn("OverTime", when(col("OverTime") == "Yes", True).otherwise(False))

        # MonthlyIncome como double
        .withColumn("MonthlyIncome", col("MonthlyIncome").cast("double"))

        # PercentSalaryHike como double
        .withColumn("PercentSalaryHike", col("PercentSalaryHike").cast("double"))
)


In [0]:
# padronizar textos
from pyspark.sql.functions import lower, trim

df_silver = (
    df_silver
        .withColumn("BusinessTravel", trim(lower(col("BusinessTravel"))))
        .withColumn("Department", trim(lower(col("Department"))))
        .withColumn("EducationField", trim(lower(col("EducationField"))))
        .withColumn("Gender", trim(lower(col("Gender"))))
)


In [0]:
# tratar valores nulos
df_silver = df_silver.fillna({
    "EnvironmentSatisfaction": 0,
    "JobSatisfaction": 0,
    "WorkLifeBalance": 0
})


In [0]:
#Renomear todas as colunas para snake_case
import re

def to_snake_case(col_name):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', col_name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1)
    return s2.lower()

for col_name in df_silver.columns:
    df_silver = df_silver.withColumnRenamed(col_name, to_snake_case(col_name))


In [0]:
#Criar a tabela Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_hr_attrition")


In [0]:
#Validar a Silver
spark.table("silver_hr_attrition").display()


age,attrition,business_travel,daily_rate,department,distance_from_home,education,education_field,employee_count,employee_number,environment_satisfaction,gender,hourly_rate,job_involvement,job_level,job_role,job_satisfaction,marital_status,monthly_income,monthly_rate,num_companies_worked,over18,over_time,percent_salary_hike,performance_rating,relationship_satisfaction,standard_hours,stock_option_level,total_working_years,training_times_last_year,work_life_balance,years_at_company,years_in_current_role,years_since_last_promotion,years_with_curr_manager
41,true,travel_rarely,1102,sales,1,2,life sciences,1,1,2,female,94,3,2,Sales Executive,4,Single,5993.0,19479,8,Y,true,11.0,3,1,80,0,8,0,1,6,4,0,5
49,false,travel_frequently,279,research & development,8,1,life sciences,1,2,3,male,61,2,2,Research Scientist,2,Married,5130.0,24907,1,Y,false,23.0,4,4,80,1,10,3,3,10,7,1,7
37,true,travel_rarely,1373,research & development,2,2,other,1,4,4,male,92,2,1,Laboratory Technician,3,Single,2090.0,2396,6,Y,true,15.0,3,2,80,0,7,3,3,0,0,0,0
33,false,travel_frequently,1392,research & development,3,4,life sciences,1,5,4,female,56,3,1,Research Scientist,3,Married,2909.0,23159,1,Y,true,11.0,3,3,80,0,8,3,3,8,7,3,0
27,false,travel_rarely,591,research & development,2,1,medical,1,7,1,male,40,3,1,Laboratory Technician,2,Married,3468.0,16632,9,Y,false,12.0,3,4,80,1,6,3,3,2,2,2,2
32,false,travel_frequently,1005,research & development,2,2,life sciences,1,8,4,male,79,3,1,Laboratory Technician,4,Single,3068.0,11864,0,Y,false,13.0,3,3,80,0,8,2,2,7,7,3,6
59,false,travel_rarely,1324,research & development,3,3,medical,1,10,3,female,81,4,1,Laboratory Technician,1,Married,2670.0,9964,4,Y,true,20.0,4,1,80,3,12,3,2,1,0,0,0
30,false,travel_rarely,1358,research & development,24,1,life sciences,1,11,4,male,67,3,1,Laboratory Technician,3,Divorced,2693.0,13335,1,Y,false,22.0,4,2,80,1,1,2,3,1,0,0,0
38,false,travel_frequently,216,research & development,23,3,life sciences,1,12,4,male,44,2,3,Manufacturing Director,3,Single,9526.0,8787,0,Y,false,21.0,4,2,80,0,10,2,3,9,7,1,8
36,false,travel_rarely,1299,research & development,27,3,medical,1,13,3,male,94,3,2,Healthcare Representative,3,Married,5237.0,16577,6,Y,false,13.0,3,2,80,2,17,3,2,7,7,7,7
